# Continuous Delivery with Gitea Actions

In this notebook, we look at how Continuous Delivery (CD) has been set up on Gitea for `Flixtube`.

**Note that you must have completed the previous notebook (01_gitea_flixtube_repository) before working through this notebook.**

---

## Examine the Gitea Workflows

- Visit http://192.168.122.99:3000/flixtube/flixtube/actions
  - Notice the workflows in the left margin, under `All Workflows`.
    - Notice the workflows that end in `ci` (these are part of the automated CI of the CI/CD  pipepline).
    - Notice the workflows that end in `cd` (these are part of the manual CD of the CI/CD pipepline).

### The Continuous Delivery (CD) Workflow for `web-cd.yaml`

- Let's examine the `web-cd.yaml` workflow.
  - Click on `Code` (top left), then on `.gitea/workflows`, and finally on `web-cd.yaml`.
  - Notice that this file is located in the folder `.gitea/workflows`.
    - **All Gitea workflows have to be located in this folder**.
  - In the main area, the contents of the file is displayed.

- Let's open the file in VSCode instead.
  - Right-click the file `flixtube/.gitea/workflows/web-cd.yaml`, choose `Open to the side`, and look at the various commads in the file.
  - `name` gives the workflow a name (displayed under Gitea Actions).
  - `on` controls if and when the workflow is triggered.
    - `workflow_dispatch` means this workflow can be triggered manually from GitHub's GUI.
    - Notice that `push` or `pull_request` are commented out for this workflow, so the workflow can only be triggered manually.
  - `jobs` is used to define one or more jobs (a workflow is made up of one or more jobs).
    - `deploy` is the name of the one and only job defined in this workflow.
      - `runs-on` determines what image is used to execute the *runner* on.
        - The current setting `ubuntu-latest` means an Ubuntu-based container will be used to run all commands in.
      - `env` is used to define environment variables available to all `steps` in the job.
        - In this case numerous environment variables are defined.
        - Notice many environment variables' values are set using *Gitea variables*.
          - A *Gitea variable's* value is obtained by placing it between double curly braces, preceeded by a dollar sign `$`.
          - `secrets.<NAME>` is either a `SECRET` or a `VARIABLE` defined under `Settings -> Actions -> Secrets` or `Settings -> Actions -> Variables` on Gitea.
          - `gitea.sha` will return the Secure Hasing Algorithm (SHA) hash for the latest Git commit that trigged the workflow.
      - `steps` contains a list of steps (tasks) that will be executed sequentailly.
        - `name` is used to name a step.
        - `uses` executes a pre-defined action.
          - Most pre-defined actions can be found here: https://github.com/actions
          - `with` configures a pre-defined action with name-value pairs specific to the action.
        - `run` is used to run a command in the container (based on `ubuntu-latest` in this case).
      - The current steps do the following.
        - `uses: actions/checkout@v3` (https://github.com/actions/checkout) checks out the Gitea repository to the *runner's* container (i.e. copies the Gitea repository to the container based on `ubuntu-latest`).
        - `run:| sudo apt-get update sudo apt-get install -y gettext` updates the package index and then installs the package `gettext` that contains the command `envsubst` which is used by the scripts in later steps.
        - `run: chmod +x ./scripts/cicd/build-image.sh && ./scripts/cicd/build-image.sh` executes the given command in the *runner's* container.
          - This will make the script `build-image.sh` executable.
          - Then the script `build-image.sh` is executed in the *runner's* container.
          - If you look at the file `scripts/cicd/build-image.sh` you'll see that:
            - It uses four of the environment variables set above under `env`.
            -  It runs `docker build` to build the `Flixtube.Web` image.
        - `run: chmod +x ./scripts/cicd/push-image.sh && ./scripts/cicd/push-image.sh` executes the given command in the *runner's* container.
          - This will make the script `push-image.sh` executable.
          - Then the script `push-image.sh` is executed in the *runner's* container.
          - If you look at the file `scripts/cicd/push-image.sh` you'll see that:
            - It uses five of the environment variables set above under `env`.
            - It runs `docker login` to login to the Azure Container Registry. 
            -  It runs `docker push` to push the `Flixtube.Web` image to the Azure Container Registry.
        - `uses: tale/kubectl-action@v1` (https://github.com/tale/kubectl-action) installs `kubectl` to  the *runner's* conatiner and connects to the Azure Kubernetes Services (AKS) cluster.
          - In this case, it `kubectl` version `1.24.2`.
          - Also notice that is uses a specific kube config from a Gitea Actions SECRET.
        - `run: chmod +x ./scripts/cicd/deploy.sh && ./scripts/cicd/deploy.sh` executes the given command in the *runner's* container.
          - This will make the script `deploy.sh` executable.
          - Then the script `deploy.sh` is executed in the *runner's* container.
          - If you look at the file `scripts/cicd/deploy.sh` you'll see that:
            - It uses six of the environment variables set above under `env`. 
            - It runs `kubectl apply` using the microservices Kubenetes YAML manifest file to deploy the microservice to the Azure Kubernetes Services (AKS) cluster.

- **Note that the other CD workflow files have a similar structure.**

## Deploy the Microservices to the Kubernetes Cluster

### Create a namespace `flixtube` in the Kubernetes cluster

- The microservices will be deployed to their own Kubernetes namespace `flixtube`.
- Run the command in the cell below to create the `flixtube` namespace.

In [1]:
%%bash

export KUBECONFIG=/home/student/.kube/config-u24s02
kubectl create namespace flixtube

namespace/flixtube created


### Deploy Backing Services and Microservices

- To deploy our `Flixtube` application to the Kubernetes cluster, we need to deploy:
  - The backing services:
    - `rabbitmq-cd.yaml`
    - `sqlserver-cd`
    - `minio-cd.yaml`
      - Note, if we were deploying to Azure storage, we would not deploy `minio-cd.yaml`.
  - Our own microservices:
    - `minio-storage-cd.yaml`
      - Note, if we were deploying to Azure storage, we would deploy `azure-storage-cd.yaml` instead.
    - `metadata-cd.yaml`
    - `history-cd.yaml`
    - `video-upload-cd.yaml`
    - `video-streaming-cd.yaml`
    - `gateway-cd.yaml`
    - `web-cd.yaml`

### Deploy the Microservices via the Gitea User Interface

- Let's trigger the CD workflows using the Gitea Web UI.
  - Visit `http://192.168.122.99:3000/flixtube/flixtube/actions` which takes us to the `Actions` tab.
  - In the left margin, under `All Workflows`:
    - Click on `rabbitmq-cd.yaml`, then the `Run Workflow` drop-down listbox, then the `Run Workflow` button.
      - Click the `Actions` tab (near the top) and notice the workflow has been triggered.
      - Wait until the icon turns green, for a successful deployment. 
    - Repeat the steps above for the CD workflows below (in order).
      - `sqlserver-cd.yaml`
      - `minio-cd.yaml`
      - `minio-storage-cd.yaml`
      - `metadata-cd.yaml`
      - `history-cd.yaml`
      - `video-upload-cd.yaml`
      - `video-streaming-cd.yaml`
      - `gateway-cd.yaml`
      - `web-cd.yaml`

---

## List Kuberenetes Resources

- Wait until all CD workflows are green in the Gitea Web UI.
- Then run the cell below to list the resources (pods, deployments and services) in the Kubernetes cluster.
- Notice all pods are up and running.
  - The pod named `db` is the `SqlServer` backing service (the others are self-explanatory).

In [4]:
%%bash

export KUBECONFIG=/home/student/.kube/config-u24s02
kubectl get pods,deployments,services -n flixtube

NAME                                 READY   STATUS      RESTARTS   AGE
pod/db-57f887f5bc-htssl              1/1     Running     0          5m45s
pod/gateway-79b65cd699-fpz6k         1/1     Running     0          78s
pod/history-6cffccb5-hsjdv           1/1     Running     0          2m55s
pod/metadata-5d57ffb784-rb2fm        1/1     Running     0          3m43s
pod/minio-68976ff65-5p5bc            1/1     Running     0          5m28s
pod/minio-mc-hfl5r                   0/1     Completed   0          5m28s
pod/rabbit-b9f74fdfb-dk6sn           1/1     Running     0          7m35s
pod/video-storage-6b4586554d-sd5cz   1/1     Running     0          4m32s
pod/video-streaming-8855dd97-2gqsh   1/1     Running     0          110s
pod/video-upload-584f8b5fdc-lf27c    1/1     Running     0          2m21s
pod/web-7cb6d8d78b-slzwg             1/1     Running     0          32s

NAME                              READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/db                1/1     1    

---

## Test the Flixtube Application in Kubernetes Cluster

**Note that It might take a while before the Flixtube web page becomes available.**

- Open a new terminal and run the command below to port-forward to the Web service.

  ```bash
  export KUBECONFIG=/home/student/.kube/config-u24s02
  kubectl port-forward deploy/web -n flixtube 4000:4000
  ```

- Open another new terminal and run the command below to port-forward to the Gateway service.

  ```bash
  export KUBECONFIG=/home/student/.kube/config-u24s02
  kubectl port-forward deploy/gateway -n flixtube 4010:4010
  ```

- Visit http://localhost:4000
  - Click the `Upload Video` button.
  - Click the `Choose File` button and select a video file.
    - You can upload the sample video file `SampleVideo_1280x720_1mb.mp4` in the folder `workshop5/02_Continuous_Integration_and_Delivery`.
  - Click `Upload`.
  - Back on the Home Page, click the `Play` button to view the video.
  - Click the `Home` button when done.
  - Back on the Home Page, click the `Show Viewing History` button to show the viewing history.
  - Click the `Home` button when done.
  - Back on the Home Page, click the `Delete` button to delete the video.
  - Confirm the deletion by clicking the `Delete` button.
- Visit http://localhost:4010/api/history
  - You should see a JSON document with the viewing history.

- Press `Ctrl + C` in the two terminals to stop port-forwarding.

---

## Clean up

### Destroy the Infrastructure with Terraform Destroy

- Let's destroy our infrastructure (i.e. delete the two VMs running Gitea and Kubernetes) by running `terraform destroy`.
- Let's also delete the `flixtube` folder in `Workshop5/02_Continuous_Integration_and_Delivery`.
- Run the cell below to destroy (delete) the two VMs and delete the `flixtube` folder.

In [20]:
%%bash

rm -rf flixtube

cd ../01_Infrastructure_as_Code/terraform/ubuntu-server-24
terraform destroy -auto-approve

cd ../ubuntu-server-24-2
terraform destroy -auto-approve

libvirt_cloudinit_disk.cidata: Refreshing state... [id=3844b26ca575adfd]
libvirt_volume.disk: Refreshing state... [id=/var/lib/libvirt/images/u24s01.qcow2]
libvirt_volume.cidata_iso: Refreshing state... [id=/var/lib/libvirt/images/u24s01-cidata.iso]
libvirt_domain.vm: Refreshing state... [name=u24s01]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # libvirt_cloudinit_disk.cidata will be destroyed
  - resource "libvirt_cloudinit_disk" "cidata" {
      - id             = "3844b26ca575adfd" -> null
      - meta_data      = <<-EOT
            instance-id: u24s01
            local-hostname: u24s01
        EOT -> null
      - name           = "u24s01-cidata" -> null
      - network_config = <<-EOT
            network:
              version: 2
              renderer: networkd
              ethernets:
                enp1s0:
                